In [5]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: Tesla T4


In [6]:
%%writefile model_ctd_tcn.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class Chomp1d(nn.Module):
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x[:, :, :-self.chomp_size].contiguous() if self.chomp_size > 0 else x


class TemporalResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int, dilation: int, dropout: float = 0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x if self.downsample is None else self.downsample(x)

        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.drop1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.drop2(out)

        return self.relu(out + residual)


class SharedTCNEncoder(nn.Module):
    def __init__(self, in_features: int = 5, hidden_dims: list = [32, 64, 64], kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        layers = []
        for i in range(len(hidden_dims)):
            dilation_size = 2 ** i
            in_ch = in_features if i == 0 else hidden_dims[i - 1]
            out_ch = hidden_dims[i]
            layers.append(TemporalResidualBlock(in_ch, out_ch, kernel_size, stride=1, dilation=dilation_size, dropout=dropout))

        self.network = nn.Sequential(*layers)
        self.latent_dim = hidden_dims[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input shape: (Batch, Channels=5, Sequence_Length)
        features = self.network(x)
        return torch.mean(features, dim=2)


class CognitiveTaskDivergenceNet(nn.Module):
    def __init__(self, in_features: int = 5, stat_features: int = 30, hidden_dims: list = [32, 64, 64],
                 dropout: float = 0.25, use_divergence: bool = True):
        super().__init__()
        self.use_divergence = use_divergence
        self.encoder = SharedTCNEncoder(in_features=in_features, hidden_dims=hidden_dims, dropout=dropout)
        latent_dim = self.encoder.latent_dim

        if self.use_divergence:
            fused_dim = (latent_dim * 4) + stat_features
        else:
            fused_dim = (latent_dim * 3) + stat_features

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, x_t1: torch.Tensor, x_t4: torch.Tensor, x_t5: torch.Tensor, x_stat: torch.Tensor):
        z1 = self.encoder(x_t1)
        z4 = self.encoder(x_t4)
        z5 = self.encoder(x_t5)

        if self.use_divergence:
            delta_phono = torch.abs(z5 - z4)
            delta_sublex = torch.abs(z1 - z4)
            fused = torch.cat([z4, z5, delta_phono, delta_sublex, x_stat], dim=-1)
        else:
            fused = torch.cat([z1, z4, z5, x_stat], dim=-1)

        logits = self.classifier(fused).squeeze(-1)
        return logits, delta_phono if self.use_divergence else None

Writing model_ctd_tcn.py


In [7]:
%%writefile losses.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class CTDLoss(nn.Module):
    def __init__(self, lambda_reg: float = 0.1, margin: float = 1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.lambda_reg = lambda_reg
        self.margin = margin

    def forward(self, logits: torch.Tensor, targets: torch.Tensor, delta_phono: torch.Tensor = None):
        cls_loss = self.bce(logits, targets)
        if delta_phono is None or self.lambda_reg == 0.0:
            return cls_loss, cls_loss, torch.tensor(0.0)

        norm_delta = torch.norm(delta_phono, p=2, dim=1)
        reg_loss = (1.0 - targets) * torch.pow(norm_delta, 2) + \
                   targets * torch.pow(F.relu(self.margin - norm_delta), 2)
        reg_loss = torch.mean(reg_loss)

        total_loss = cls_loss + (self.lambda_reg * reg_loss)
        return total_loss, cls_loss, reg_loss

Writing losses.py


In [12]:
import os
import zipfile
import glob
import pandas as pd
import numpy as np

# -------------------------------------------------------------
# 1. UNZIP DATASET FROM COLAB STORAGE
# -------------------------------------------------------------
zip_path = "/dataset.zip" if os.path.exists("/dataset.zip") else "/content/dataset.zip"
extract_dir = "/content/etdd70_extracted"

if not os.path.exists(extract_dir) or len(os.listdir(extract_dir)) == 0:
    print(f"Extracting {zip_path} to {extract_dir}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete.")
else:
    print(f"Using existing extracted files in {extract_dir}.")

# If the zip created a single nested parent directory, step into it
extracted_items = [os.path.join(extract_dir, d) for d in os.listdir(extract_dir) if not d.startswith('.')]
if len(extracted_items) == 1 and os.path.isdir(extracted_items[0]):
    DATA_DIR = extracted_items[0]
else:
    DATA_DIR = extract_dir

print(f"Active Dataset Directory: {DATA_DIR}")

# -------------------------------------------------------------
# 2. LOCATE METADATA / LABELS
# -------------------------------------------------------------
# Search recursively for any metadata/participant CSV file
possible_metadata = glob.glob(os.path.join(DATA_DIR, "**", "*participant*.csv"), recursive=True) + \
                    glob.glob(os.path.join(DATA_DIR, "**", "*metadata*.csv"), recursive=True) + \
                    glob.glob(os.path.join(DATA_DIR, "**", "*label*.csv"), recursive=True)

if possible_metadata:
    metadata_file = possible_metadata[0]
    print(f"Found metadata file: {metadata_file}")
    meta_df = pd.read_csv(metadata_file)

    # Auto-detect subject ID and label column names
    id_col = next((c for c in meta_df.columns if any(k in c.lower() for k in ['id', 'subject', 'participant'])), meta_df.columns[0])
    lbl_col = next((c for c in meta_df.columns if any(k in c.lower() for k in ['label', 'dyslex', 'class', 'group'])), meta_df.columns[-1])

    # Ensure binary format (0 for typical, 1 for dyslexic)
    raw_labels = meta_df[lbl_col]
    if raw_labels.dtype == object:
        unique_vals = raw_labels.unique()
        dys_val = next(v for v in unique_vals if any(d in str(v).lower() for d in ['dys', '1', 'true', 'yes']))
        labels = (raw_labels == dys_val).astype(np.float32).to_numpy()
    else:
        labels = raw_labels.to_numpy(dtype=np.float32)

    subject_ids = meta_df[id_col].astype(str).tolist()
else:
    # Fallback: scan for numbered subject subdirectories directly
    subject_dirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith('.')]
    subject_dirs.sort()
    subject_ids = subject_dirs
    print(f"No metadata CSV detected. Inferred {len(subject_ids)} subject folders.")
    # Default placeholder if labels are in separate txt/csv files
    labels = np.zeros(len(subject_ids), dtype=np.float32)

# -------------------------------------------------------------
# 3. EXTRACT FIXATION SEQUENCES & STATISTICAL METRICS
# -------------------------------------------------------------
FIXATION_COLS = ['duration_ms', 'fix_x', 'fix_y', 'disp_x', 'disp_y']

raw_t1_list, raw_t4_list, raw_t5_list = [], [], []
stats_list = []
valid_labels = []

def find_file(sub_folder, patterns):
    for pat in patterns:
        matches = glob.glob(os.path.join(sub_folder, f"*{pat}*"))
        if matches:
            return matches[0]
    return None

def extract_fixation_array(file_path):
    df = pd.read_csv(file_path)
    # Check if exact expected columns exist
    if all(col in df.columns for col in FIXATION_COLS):
        return df[FIXATION_COLS].dropna().to_numpy()
    # If column names differ, take the first 5 numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns[:5]
    return df[numeric_cols].dropna().to_numpy()

print(f"Processing subjects...")
for idx, sub in enumerate(subject_ids):
    # Match subject directory
    sub_folder = os.path.join(DATA_DIR, sub)
    if not os.path.exists(sub_folder):
        matches = glob.glob(os.path.join(DATA_DIR, "**", sub), recursive=True)
        if matches:
            sub_folder = matches[0]
        else:
            continue

    # Locate T1, T4, T5 fixation logs
    f_t1 = find_file(sub_folder, ['T1', 't1', 'syllable'])
    f_t4 = find_file(sub_folder, ['T4', 't4', 'meaningful', 'text'])
    f_t5 = find_file(sub_folder, ['T5', 't5', 'pseudo'])
    f_stat = find_file(sub_folder, ['stat', 'metric', 'summary'])

    if not (f_t1 and f_t4 and f_t5):
        continue

    seq_t1 = extract_fixation_array(f_t1)
    seq_t4 = extract_fixation_array(f_t4)
    seq_t5 = extract_fixation_array(f_t5)

    raw_t1_list.append(seq_t1)
    raw_t4_list.append(seq_t4)
    raw_t5_list.append(seq_t5)

    # Statistical features (10 metrics x 3 tasks = 30)
    if f_stat:
        df_st = pd.read_csv(f_stat).select_dtypes(include=[np.number])
        stat_vals = df_st.to_numpy().flatten()
        if len(stat_vals) < 30:
            stat_vals = np.pad(stat_vals, (0, 30 - len(stat_vals)), mode='constant')
        else:
            stat_vals = stat_vals[:30]
    else:
        # Fallback to mean/std summary if no standalone stats file exists
        stat_vals = np.concatenate([
            np.mean(seq_t1, axis=0), np.std(seq_t1, axis=0),
            np.mean(seq_t4, axis=0), np.std(seq_t4, axis=0),
            np.mean(seq_t5, axis=0), np.std(seq_t5, axis=0)
        ])[:30]

    stats_list.append(stat_vals)
    valid_labels.append(labels[idx])

# -------------------------------------------------------------
# 4. TRUNCATION, PADDING, AND TENSOR FORMATTING
# -------------------------------------------------------------
data_stats = np.array(stats_list, dtype=np.float32)
labels = np.array(valid_labels, dtype=np.float32)

all_lengths = [len(seq) for seq in (raw_t1_list + raw_t4_list + raw_t5_list)]
max_seq_len = int(np.percentile(all_lengths, 75))
print(f"\nSequence length selected (75th percentile): {max_seq_len}")

def pad_and_format_sequences(raw_seqs, max_len):
    N = len(raw_seqs)
    formatted = np.zeros((N, max_len, 5), dtype=np.float32)
    for i, seq in enumerate(raw_seqs):
        length = min(len(seq), max_len)
        formatted[i, :length, :] = seq[:length, :5]
    # Output shape for PyTorch Conv1D: (N, Channels=5, Length)
    return np.transpose(formatted, (0, 2, 1))

data_t1 = pad_and_format_sequences(raw_t1_list, max_seq_len)
data_t4 = pad_and_format_sequences(raw_t4_list, max_seq_len)
data_t5 = pad_and_format_sequences(raw_t5_list, max_seq_len)

print("\n" + "=" * 40)
print("       ARRAYS READY FOR STEP 4          ")
print("=" * 40)
print(f"data_t1 shape:    {data_t1.shape}    (Subjects, 5 Channels, Length)")
print(f"data_t4 shape:    {data_t4.shape}    (Subjects, 5 Channels, Length)")
print(f"data_t5 shape:    {data_t5.shape}    (Subjects, 5 Channels, Length)")
print(f"data_stats shape: {data_stats.shape} (Subjects, 30 Metrics)")
print(f"labels shape:     {labels.shape}     (Subjects,)")
print("=" * 40)

Extracting /content/dataset.zip to /content/etdd70_extracted...


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset.zip'

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np

# 1. Locate the data folder containing the 840 files
candidate_dirs = [
    "/content/etdd70_extracted/data",
    "/content/etdd70_extracted",
    "/content/data"
]
DATA_DIR = next((d for d in candidate_dirs if os.path.exists(d) and len(glob.glob(os.path.join(d, "Subject_*"))) > 0), None)

if DATA_DIR is None:
    raise FileNotFoundError("Could not locate folder with Subject_* files.")

print(f"Loading files from: {DATA_DIR}")

# 2. Extract unique Subject IDs from the T1 fixation files
t1_files = sorted(glob.glob(os.path.join(DATA_DIR, "Subject_*_T1_Syllables_fixations.csv")))
subject_ids = [re.search(r"Subject_(\d+)_", f).group(1) for f in t1_files]
num_subjects = len(subject_ids)
print(f"Successfully identified {num_subjects} subjects.")

# 3. Inspect a sample metrics file to locate features and label column
sample_metrics_file = os.path.join(DATA_DIR, f"Subject_{subject_ids[0]}_T1_Syllables_metrics.csv")
sample_df = pd.read_csv(sample_metrics_file)
print(f"Metrics columns detected: {list(sample_df.columns)}")

# Identify label column (e.g., 'dyslexia', 'label', 'class', 'group')
possible_label_cols = [c for c in sample_df.columns if any(k in c.lower() for k in ['dyslex', 'label', 'class', 'group', 'diag'])]
label_col = possible_label_cols[0] if possible_label_cols else None
print(f"Identified label column: '{label_col}'")

# 4. Extract Fixations, Statistical Metrics, and Labels across all 70 subjects
raw_t1_list, raw_t4_list, raw_t5_list = [], [], []
stats_list = []
labels_list = []

for sub_id in subject_ids:
    # A. Fixation sequences
    f_t1 = os.path.join(DATA_DIR, f"Subject_{sub_id}_T1_Syllables_fixations.csv")
    f_t4 = os.path.join(DATA_DIR, f"Subject_{sub_id}_T4_Meaningful_Text_fixations.csv")
    f_t5 = os.path.join(DATA_DIR, f"Subject_{sub_id}_T5_Pseudo_Text_fixations.csv")

    # Read first 5 numeric columns: [dur, fix_x, fix_y, disp_x, disp_y]
    df_t1 = pd.read_csv(f_t1).select_dtypes(include=[np.number]).dropna()
    df_t4 = pd.read_csv(f_t4).select_dtypes(include=[np.number]).dropna()
    df_t5 = pd.read_csv(f_t5).select_dtypes(include=[np.number]).dropna()

    raw_t1_list.append(df_t1.iloc[:, :5].to_numpy(dtype=np.float32))
    raw_t4_list.append(df_t4.iloc[:, :5].to_numpy(dtype=np.float32))
    raw_t5_list.append(df_t5.iloc[:, :5].to_numpy(dtype=np.float32))

    # B. Oculomotor statistics (10 metrics per task = 30 total)
    m_t1 = pd.read_csv(os.path.join(DATA_DIR, f"Subject_{sub_id}_T1_Syllables_metrics.csv"))
    m_t4 = pd.read_csv(os.path.join(DATA_DIR, f"Subject_{sub_id}_T4_Meaningful_Text_metrics.csv"))
    m_t5 = pd.read_csv(os.path.join(DATA_DIR, f"Subject_{sub_id}_T5_Pseudo_Text_metrics.csv"))

    # Extract label from the first task's metrics if available
    if label_col:
        lbl_val = m_t1[label_col].iloc[0]
        # Map string/numeric label to binary 0 or 1
        labels_list.append(1.0 if str(lbl_val).lower() in ['1', 'true', 'yes', 'dyslexic', 'dyslexia'] else 0.0)

    # Exclude non-metric or label columns from statistical features
    cols_to_drop = [label_col] if label_col else []
    cols_to_drop += [c for c in m_t1.columns if any(k in c.lower() for k in ['id', 'subject', 'name'])]

    s1 = m_t1.drop(columns=[c for c in cols_to_drop if c in m_t1.columns]).select_dtypes(include=[np.number]).to_numpy().flatten()
    s4 = m_t4.drop(columns=[c for c in cols_to_drop if c in m_t4.columns]).select_dtypes(include=[np.number]).to_numpy().flatten()
    s5 = m_t5.drop(columns=[c for c in cols_to_drop if c in m_t5.columns]).select_dtypes(include=[np.number]).to_numpy().flatten()

    combined_stats = np.concatenate([s1[:10], s4[:10], s5[:10]])
    if len(combined_stats) < 30:
        combined_stats = np.pad(combined_stats, (0, 30 - len(combined_stats)), mode='constant')
    stats_list.append(combined_stats[:30])

# 5. Format sequence lengths (75th percentile padding/truncation)
all_lengths = [len(seq) for seq in (raw_t1_list + raw_t4_list + raw_t5_list)]
max_seq_len = int(np.percentile(all_lengths, 75))
print(f"Target sequence length (75th percentile): {max_seq_len}")

def pad_and_format_sequences(raw_seqs, max_len):
    N = len(raw_seqs)
    formatted = np.zeros((N, max_len, 5), dtype=np.float32)
    for i, seq in enumerate(raw_seqs):
        length = min(len(seq), max_len)
        formatted[i, :length, :] = seq[:length, :5]
    # Shape for PyTorch Conv1d: (N, Channels=5, Length)
    return np.transpose(formatted, (0, 2, 1))

data_t1 = pad_and_format_sequences(raw_t1_list, max_seq_len)
data_t4 = pad_and_format_sequences(raw_t4_list, max_seq_len)
data_t5 = pad_and_format_sequences(raw_t5_list, max_seq_len)
data_stats = np.array(stats_list, dtype=np.float32)
labels = np.array(labels_list, dtype=np.float32) if labels_list else np.zeros(num_subjects, dtype=np.float32)

print("\n" + "=" * 50)
print("             DATA PREPROCESSING READY             ")
print("=" * 50)
print(f"data_t1 shape:    {data_t1.shape}    (70, 5, Length)")
print(f"data_t4 shape:    {data_t4.shape}    (70, 5, Length)")
print(f"data_t5 shape:    {data_t5.shape}    (70, 5, Length)")
print(f"data_stats shape: {data_stats.shape} (70, 30)")
print(f"labels shape:     {labels.shape}     (70,)")
print(f"Class breakdown:  Dyslexic = {int(np.sum(labels))}, Non-Dyslexic = {int(len(labels) - np.sum(labels))}")
print("=" * 50)

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.svm import SVC

ablation_configs = [
    {"name": "Full CTD-TCN (Proposed)", "use_div": True, "use_stats": True, "tasks": ["T1", "T4", "T5"]},
    {"name": "w/o Disparity (Direct Concat)", "use_div": False, "use_stats": True, "tasks": ["T1", "T4", "T5"]},
    {"name": "w/o Handcrafted Features (Deep Only)", "use_div": True, "use_stats": False, "tasks": ["T1", "T4", "T5"]},
    {"name": "w/o Pseudo-Text Task (T1 + T4 only)", "use_div": True, "use_stats": True, "tasks": ["T1", "T4"]},
    {"name": "Handcrafted Stats Only (PCA + SVM)", "model_type": "svm"}
]

ablation_results = []

for config in ablation_configs:
    cfg_name = config["name"]
    print(f"\nEvaluating: {cfg_name}...")

    y_true_cfg = []
    y_prob_cfg = []

    for fold in range(num_subjects):
        train_idx = [i for i in range(num_subjects) if i != fold]
        test_idx = [fold]

        lbl_tr = torch.tensor(labels[train_idx], dtype=torch.float32)
        lbl_te = labels[test_idx][0]

        # Classical baseline: PCA + SVM on tabular features
        if config.get("model_type") == "svm":
            s_scaler = StandardScaler().fit(data_stats[train_idx])
            s_tr = s_scaler.transform(data_stats[train_idx])
            s_te = s_scaler.transform(data_stats[test_idx])

            pca = PCA(n_components=0.90).fit(s_tr)
            svm = SVC(kernel='rbf', C=0.1, probability=True, random_state=42)
            svm.fit(pca.transform(s_tr), labels[train_idx])
            prob = svm.predict_proba(pca.transform(s_te))[0, 1]

        else:
            # Sequence Preprocessing
            t1_tr, t1_te = fit_and_scale_sequences(data_t1[train_idx], data_t1[test_idx])
            t4_tr, t4_te = fit_and_scale_sequences(data_t4[train_idx], data_t4[test_idx])
            t5_tr, t5_te = fit_and_scale_sequences(data_t5[train_idx], data_t5[test_idx])

            # Zero out inactive task sequences if testing task contribution
            if "T5" not in config["tasks"]:
                t5_tr = torch.zeros_like(t5_tr)
                t5_te = torch.zeros_like(t5_te)

            if config["use_stats"]:
                s_scaler = StandardScaler().fit(data_stats[train_idx])
                st_tr = torch.tensor(s_scaler.transform(data_stats[train_idx]), dtype=torch.float32)
                st_te = torch.tensor(s_scaler.transform(data_stats[test_idx]), dtype=torch.float32)
                n_stats = 30
            else:
                st_tr = torch.zeros((len(train_idx), 0), dtype=torch.float32)
                st_te = torch.zeros((1, 0), dtype=torch.float32)
                n_stats = 0

            train_ds = TensorDataset(t1_tr, t4_tr, t5_tr, st_tr, lbl_tr)
            loader = DataLoader(train_ds, batch_size=8, shuffle=True)

            model = CognitiveTaskDivergenceNet(
                in_features=5, stat_features=n_stats, hidden_dims=[32, 64, 64],
                dropout=0.25, use_divergence=config["use_div"]
            ).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
            criterion = CTDLoss(lambda_reg=0.1 if config["use_div"] else 0.0, margin=1.0)

            model.train()
            for epoch in range(50):
                for b_t1, b_t4, b_t5, b_st, b_y in loader:
                    b_t1, b_t4, b_t5, b_st, b_y = b_t1.to(device), b_t4.to(device), b_t5.to(device), b_st.to(device), b_y.to(device)
                    optimizer.zero_grad()
                    logits, delta = model(b_t1, b_t4, b_t5, b_st)
                    loss, _, _ = criterion(logits, b_y, delta)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                out_logits, _ = model(t1_te.to(device), t4_te.to(device), t5_te.to(device), st_te.to(device))
                prob = torch.sigmoid(out_logits).item()

        y_true_cfg.append(lbl_te)
        y_prob_cfg.append(prob)

    y_true_cfg = np.array(y_true_cfg, dtype=int)
    y_prob_cfg = np.array(y_prob_cfg)
    y_pred_cfg = (y_prob_cfg >= 0.5).astype(int)

    acc = accuracy_score(y_true_cfg, y_pred_cfg) * 100
    auc = roc_auc_score(y_true_cfg, y_prob_cfg)
    kappa = cohen_kappa_score(y_true_cfg, y_pred_cfg)
    mcc = matthews_corrcoef(y_true_cfg, y_pred_cfg)

    ablation_results.append({
        "Configuration": cfg_name,
        "Accuracy (%)": f"{acc:.2f}",
        "AUC-ROC": f"{auc:.4f}",
        "Cohen κ": f"{kappa:.4f}",
        "MCC": f"{mcc:.4f}"
    })

df_ablation = pd.DataFrame(ablation_results)
print("\n" + "="*70)
print("                    ABLATION STUDY RESULTS SUMMARY                     ")
print("="*70)
print(df_ablation.to_string(index=False))

# Export clean Markdown and LaTeX table for the paper
df_ablation.to_csv("/content/ablation_study_results.csv", index=False)
print("\nLaTeX Code for Paper Submission:\n")
print(df_ablation.to_latex(index=False))

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

# Extract latent vectors across all subjects using the trained shared backbone
model.eval()
norm_phono_list = []
norm_sublex_list = []

with torch.no_grad():
    for i in range(num_subjects):
        t1 = torch.tensor(data_t1[i:i+1], dtype=torch.float32).to(device)
        t4 = torch.tensor(data_t4[i:i+1], dtype=torch.float32).to(device)
        t5 = torch.tensor(data_t5[i:i+1], dtype=torch.float32).to(device)

        z1 = model.encoder(t1)
        z4 = model.encoder(t4)
        z5 = model.encoder(t5)

        norm_phono = torch.norm(z5 - z4, p=2, dim=1).item()
        norm_sublex = torch.norm(z1 - z4, p=2, dim=1).item()

        norm_phono_list.append(norm_phono)
        norm_sublex_list.append(norm_sublex)

norm_phono_list = np.array(norm_phono_list)
norm_sublex_list = np.array(norm_sublex_list)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)

# 1. Bivariate Latent Separation Scatter
palette = {0: '#2b5c8f', 1: '#d95f02'}
labels_text = np.array(['Typical Reader' if l == 0 else 'Dyslexic Reader' for l in labels])

for lbl_val, group_name in [(0, 'Typical Reader'), (1, 'Dyslexic Reader')]:
    idx = labels == lbl_val
    axes[0].scatter(
        norm_sublex_list[idx], norm_phono_list[idx],
        c=palette[lbl_val], label=group_name, alpha=0.8, s=65, edgecolors='none'
    )

axes[0].set_title("Cognitive Task-Divergence Latent Space", fontsize=12, pad=10)
axes[0].set_xlabel(r"Sublexical Shift: $\|\Delta z_{sublex}\| = \|z_{T1} - z_{T4}\|_2$", fontsize=11)
axes[0].set_ylabel(r"Phonological Breakdown: $\|\Delta z_{phono}\| = \|z_{T5} - z_{T4}\|_2$", fontsize=11)
axes[0].legend(frameon=True, loc='upper left')
axes[0].grid(True, alpha=0.3)

# 2. KDE Density on Phonological Shift (T5 vs T4)
sns.kdeplot(
    x=norm_phono_list[labels == 0], ax=axes[1], fill=True, color='#2b5c8f', label='Typical Readers', alpha=0.4
)
sns.kdeplot(
    x=norm_phono_list[labels == 1], ax=axes[1], fill=True, color='#d95f02', label='Dyslexic Readers', alpha=0.4
)
axes[1].set_title(r"Density of Phonological Breakdown $(\|\Delta z_{phono}\|)$", fontsize=12, pad=10)
axes[1].set_xlabel(r"Magnitude of $\|\Delta z_{phono}\|_2$", fontsize=11)
axes[1].set_ylabel("Kernel Density Estimate", fontsize=11)
axes[1].legend(frameon=True)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/cognitive_disparity_biomarker.png", dpi=300)
plt.show()
print("Saved cognitive biomarker plot to /content/cognitive_disparity_biomarker.png")

NameError: name 'model' is not defined

In [ ]:
import os
import glob
import pandas as pd

# Check metadata files in the extracted directory
all_csvs = glob.glob("/content/etdd70_extracted/**/*.csv", recursive=True)
metadata_files = [f for f in all_csvs if any(k in os.path.basename(f).lower() for k in ['meta', 'partic', 'info', 'subject', 'label'])]

print("--- Potential Metadata Files ---")
for f in metadata_files[:5]:
    print(f)

# Inspect columns and values of one subject's metrics file
sample_metrics = [f for f in all_csvs if "metrics.csv" in f][0]
print(f"\n--- Inspecting: {os.path.basename(sample_metrics)} ---")
df_sample = pd.read_csv(sample_metrics)
print("Columns in metrics file:")
print(df_sample.columns.tolist())
print("\nFirst row sample:")
display(df_sample.head(1))

In [ ]:
import requests
import json
import os

# 1. Query the ETDD70 Zenodo record cited in your paper
ZENODO_RECORD_ID = "13332134"
api_url = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

print(f"Querying Zenodo API for record {ZENODO_RECORD_ID}...")
response = requests.get(api_url)

if response.status_code == 200:
    record = response.json()

    # Handle Zenodo API formats
    files_obj = record.get("files", [])
    if isinstance(files_obj, dict):
        file_list = files_obj.get("entries", [])
    else:
        file_list = files_obj

    print(f"\nFiles available on Zenodo:")
    downloaded_files = []

    for f in file_list:
        fname = f.get("key") or f.get("filename")
        size_kb = f.get("size", 0) / 1024
        print(f" - {fname} ({size_kb:.1f} KB)")

        # Download metadata, participant, readme, or small table files
        if fname.endswith(('.csv', '.tsv', '.txt', '.xlsx', '.json')) or any(k in fname.lower() for k in ['participant', 'meta', 'subject', 'label', 'demograph']):
            links = f.get("links", {})
            dl_url = links.get("content") or links.get("self")

            if dl_url:
                print(f"   Downloading: {fname}...")
                content = requests.get(dl_url).content
                save_path = os.path.join("/content", fname)
                with open(save_path, "wb") as out:
                    out.write(content)
                downloaded_files.append(save_path)
                print(f"   Saved to: {save_path}")

    print(f"\nDownloaded {len(downloaded_files)} metadata files to /content/")
else:
    print(f"API request failed with status: {response.status_code}")

In [10]:
import glob
import pandas as pd
import numpy as np
import re

# Locate the downloaded metadata file
meta_files = [f for f in glob.glob("/content/*.*") if f.endswith(('.csv', '.tsv', '.xlsx', '.txt')) and not f.endswith('dataset.zip')]

if not meta_files:
    raise FileNotFoundError("No metadata files found. Check the output of Step 1.")

print(f"Found metadata file: {meta_files[0]}")
meta_path = meta_files[0]

# Load metadata
if meta_path.endswith('.xlsx'):
    df_meta = pd.read_excel(meta_path)
elif meta_path.endswith('.tsv') or meta_path.endswith('.txt'):
    df_meta = pd.read_csv(meta_path, sep=None, engine='python')
else:
    df_meta = pd.read_csv(meta_path)

print("\nMetadata Columns:", df_meta.columns.tolist())
display(df_meta.head(3))

# Identify ID and diagnosis columns
id_col = next((c for c in df_meta.columns if any(k in c.lower() for k in ['id', 'subject', 'participant', 'sid'])), df_meta.columns[0])
diag_col = next((c for c in df_meta.columns if any(k in c.lower() for k in ['dyslex', 'group', 'class', 'label', 'diag', 'status'])), df_meta.columns[-1])

print(f"Mapping Subject ID from '{id_col}' and Diagnosis from '{diag_col}'")

# Build dictionary
subject_label_map = {}
for _, row in df_meta.iterrows():
    raw_id = str(row[id_col])
    clean_id = re.sub(r'\D', '', raw_id)  # extract numeric ID

    val_str = str(row[diag_col]).strip().lower()
    # 1 for dyslexic, 0 for typical/control
    is_dyslexic = 1.0 if val_str in ['1', 'true', 'yes', 'd', 'dyslexia', 'dyslexic', 'positive'] else 0.0

    subject_label_map[clean_id] = is_dyslexic
    subject_label_map[raw_id] = is_dyslexic

# Map to the loaded subject sequence
labels_list = []
missing_ids = []

for sid in subject_ids:
    clean_sid = re.sub(r'\D', '', str(sid))
    if clean_sid in subject_label_map:
        labels_list.append(subject_label_map[clean_sid])
    elif str(sid) in subject_label_map:
        labels_list.append(subject_label_map[str(sid)])
    else:
        missing_ids.append(sid)

if missing_ids:
    print(f"Warning: {len(missing_ids)} subjects were not matched in metadata: {missing_ids[:5]}")
else:
    labels = np.array(labels_list, dtype=np.float32)
    print("\n" + "=" * 45)
    print("           LABELS LOADED SUCCESSFULLY         ")
    print("=" * 45)
    print(f"Total subjects: {len(labels)}")
    print(f"Dyslexic (1):   {int(np.sum(labels))}")
    print(f"Control (0):    {int(len(labels) - np.sum(labels))}")
    print("=" * 45)

FileNotFoundError: No metadata files found. Check the output of Step 1.

In [11]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_auc_score, cohen_kappa_score,
    matthews_corrcoef, confusion_matrix, roc_curve
)
from scipy.stats import binomtest
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------------------
# 1. ARCHITECTURE DEFINITIONS
# ---------------------------------------------------------------------
class Chomp1d(nn.Module):
    def __init__(self, chomp_size: int):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x[:, :, :-self.chomp_size].contiguous() if self.chomp_size > 0 else x


class TemporalResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int, dilation: int, dropout: float = 0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu1 = nn.GELU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.relu2 = nn.GELU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.relu = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x if self.downsample is None else self.downsample(x)
        out = self.drop1(self.relu1(self.bn1(self.chomp1(self.conv1(x)))))
        out = self.drop2(self.relu2(self.bn2(self.chomp2(self.conv2(out)))))
        return self.relu(out + residual)


class SharedTCNEncoder(nn.Module):
    def __init__(self, in_features: int = 5, hidden_dims: list = [32, 64, 64], kernel_size: int = 3, dropout: float = 0.2):
        super().__init__()
        layers = []
        for i in range(len(hidden_dims)):
            dilation_size = 2 ** i
            in_ch = in_features if i == 0 else hidden_dims[i - 1]
            out_ch = hidden_dims[i]
            layers.append(TemporalResidualBlock(in_ch, out_ch, kernel_size, stride=1, dilation=dilation_size, dropout=dropout))

        self.network = nn.Sequential(*layers)
        self.latent_dim = hidden_dims[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.network(x)
        return torch.mean(features, dim=2)


class CognitiveTaskDivergenceNet(nn.Module):
    def __init__(self, in_features: int = 5, stat_features: int = 30, hidden_dims: list = [32, 64, 64],
                 dropout: float = 0.25, use_divergence: bool = True):
        super().__init__()
        self.use_divergence = use_divergence
        self.encoder = SharedTCNEncoder(in_features=in_features, hidden_dims=hidden_dims, dropout=dropout)
        latent_dim = self.encoder.latent_dim

        if self.use_divergence:
            fused_dim = (latent_dim * 4) + stat_features
        else:
            fused_dim = (latent_dim * 3) + stat_features

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, x_t1: torch.Tensor, x_t4: torch.Tensor, x_t5: torch.Tensor, x_stat: torch.Tensor):
        z1 = self.encoder(x_t1)
        z4 = self.encoder(x_t4)
        z5 = self.encoder(x_t5)

        if self.use_divergence:
            delta_phono = torch.abs(z5 - z4)
            delta_sublex = torch.abs(z1 - z4)
            if x_stat.shape[1] > 0:
                fused = torch.cat([z4, z5, delta_phono, delta_sublex, x_stat], dim=-1)
            else:
                fused = torch.cat([z4, z5, delta_phono, delta_sublex], dim=-1)
        else:
            if x_stat.shape[1] > 0:
                fused = torch.cat([z1, z4, z5, x_stat], dim=-1)
            else:
                fused = torch.cat([z1, z4, z5], dim=-1)

        logits = self.classifier(fused).squeeze(-1)
        return logits, delta_phono if self.use_divergence else None


class CTDLoss(nn.Module):
    def __init__(self, lambda_reg: float = 0.1, margin: float = 1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.lambda_reg = lambda_reg
        self.margin = margin

    def forward(self, logits: torch.Tensor, targets: torch.Tensor, delta_phono: torch.Tensor = None):
        cls_loss = self.bce(logits, targets)
        if delta_phono is None or self.lambda_reg == 0.0:
            return cls_loss, cls_loss, torch.tensor(0.0)

        norm_delta = torch.norm(delta_phono, p=2, dim=1)
        reg_loss = (1.0 - targets) * torch.pow(norm_delta, 2) + \
                   targets * torch.pow(F.relu(self.margin - norm_delta), 2)
        reg_loss = torch.mean(reg_loss)

        total_loss = cls_loss + (self.lambda_reg * reg_loss)
        return total_loss, cls_loss, reg_loss


# ---------------------------------------------------------------------
# 2. LEAKAGE-FREE LOO-CV ENGINE
# ---------------------------------------------------------------------
def fit_and_scale_sequences(seq_train, seq_test):
    N_tr, C, L = seq_train.shape
    flat_train = seq_train.transpose(0, 2, 1).reshape(-1, C)
    scaler = StandardScaler().fit(flat_train)

    scaled_tr = scaler.transform(flat_train).reshape(N_tr, L, C).transpose(0, 2, 1)
    flat_te = seq_test.transpose(0, 2, 1).reshape(-1, C)
    scaled_te = scaler.transform(flat_te).reshape(1, L, C).transpose(0, 2, 1)

    return torch.tensor(scaled_tr, dtype=torch.float32), torch.tensor(scaled_te, dtype=torch.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing CTD-TCN LOO-CV on device: {device}")

num_subjects = len(labels)
y_true_all = []
y_pred_probs = []

torch.manual_seed(42)
np.random.seed(42)

for fold in range(num_subjects):
    train_idx = [i for i in range(num_subjects) if i != fold]
    test_idx = [fold]

    # Preprocessing strictly fitted on 69 training subjects
    t1_tr, t1_te = fit_and_scale_sequences(data_t1[train_idx], data_t1[test_idx])
    t4_tr, t4_te = fit_and_scale_sequences(data_t4[train_idx], data_t4[test_idx])
    t5_tr, t5_te = fit_and_scale_sequences(data_t5[train_idx], data_t5[test_idx])

    stat_scaler = StandardScaler().fit(data_stats[train_idx])
    stat_tr = torch.tensor(stat_scaler.transform(data_stats[train_idx]), dtype=torch.float32)
    stat_te = torch.tensor(stat_scaler.transform(data_stats[test_idx]), dtype=torch.float32)

    lbl_tr = torch.tensor(labels[train_idx], dtype=torch.float32)
    lbl_te = labels[test_idx][0]

    train_loader = DataLoader(TensorDataset(t1_tr, t4_tr, t5_tr, stat_tr, lbl_tr), batch_size=8, shuffle=True)

    # Re-initialize clean weights per fold
    model = CognitiveTaskDivergenceNet(
        in_features=5, stat_features=30, hidden_dims=[32, 64, 64], dropout=0.25, use_divergence=True
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    criterion = CTDLoss(lambda_reg=0.1, margin=1.0)

    model.train()
    for epoch in range(50):
        for b_t1, b_t4, b_t5, b_st, b_y in train_loader:
            b_t1, b_t4, b_t5, b_st, b_y = b_t1.to(device), b_t4.to(device), b_t5.to(device), b_st.to(device), b_y.to(device)
            optimizer.zero_grad()
            logits, delta = model(b_t1, b_t4, b_t5, b_st)
            loss, _, _ = criterion(logits, b_y, delta)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

    # Held-out single subject test
    model.eval()
    with torch.no_grad():
        out_logits, _ = model(t1_te.to(device), t4_te.to(device), t5_te.to(device), stat_te.to(device))
        prob = torch.sigmoid(out_logits).item()

    y_true_all.append(lbl_te)
    y_pred_probs.append(prob)

    if (fold + 1) % 10 == 0 or fold == num_subjects - 1:
        print(f"Processed Fold [{fold + 1:02d}/{num_subjects:02d}]")

# ---------------------------------------------------------------------
# 3. METRICS & PUBLICATION FIGURES
# ---------------------------------------------------------------------
y_true_all = np.array(y_true_all, dtype=int)
y_pred_probs = np.array(y_pred_probs)
y_pred_labels = (y_pred_probs >= 0.5).astype(int)

fpr, tpr, thresholds = roc_curve(y_true_all, y_pred_probs)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

cm = confusion_matrix(y_true_all, y_pred_labels)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) * 100
specificity = tn / (tn + fp) * 100
accuracy = (tp + tn) / len(y_true_all) * 100
auc_roc = roc_auc_score(y_true_all, y_pred_probs)
kappa = cohen_kappa_score(y_true_all, y_pred_labels)
mcc = matthews_corrcoef(y_true_all, y_pred_labels)
binom_res = binomtest(int(tp + tn), n=num_subjects, p=0.5, alternative='greater')

print("\n" + "=" * 50)
print("            CTD-TCN LOO-CV FINAL RESULTS          ")
print("=" * 50)
print(f"Accuracy:            {accuracy:.2f}% ({tp + tn}/{num_subjects})")
print(f"AUC-ROC:             {auc_roc:.4f}")
print(f"Sensitivity:         {sensitivity:.2f}% ({tp}/{tp + fn})")
print(f"Specificity:         {specificity:.2f}% ({tn}/{tn + fp})")
print(f"Cohen's Kappa (κ):   {kappa:.4f}")
print(f"Matthews CC (MCC):   {mcc:.4f}")
print(f"Binomial Test p-val: {binom_res.pvalue:.4e}")
print(f"Optimal Threshold:   {optimal_threshold:.4f}")
print("=" * 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
    xticklabels=['Non-Dyslexic', 'Dyslexic'],
    yticklabels=['Non-Dyslexic', 'Dyslexic'],
    annot_kws={"fontsize": 14, "weight": "bold"}
)
axes[0].set_title(f"Confusion Matrix (Acc={accuracy:.1f}%, Sens={sensitivity:.1f}%, Spec={specificity:.1f}%)", fontsize=12, pad=10)
axes[0].set_xlabel("Predicted Label", fontsize=11)
axes[0].set_ylabel("True Label", fontsize=11)

axes[1].plot(fpr, tpr, color='#1f77b4', lw=2.5, label=f'CTD-TCN (AUC = {auc_roc:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5, label='Random Baseline')
axes[1].scatter(fpr[optimal_idx], tpr[optimal_idx], color='crimson', s=70, zorder=5, label=f'Optimal Th = {optimal_threshold:.2f}')
axes[1].set_xlim([-0.02, 1.02])
axes[1].set_ylim([-0.02, 1.05])
axes[1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
axes[1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
axes[1].set_title(f"ROC Curve (κ = {kappa:.4f}, MCC = {mcc:.4f})", fontsize=12, pad=10)
axes[1].legend(loc="lower right", frameon=True)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/ctd_tcn_performance.png", dpi=300)
plt.show()
print("Saved performance plot to /content/ctd_tcn_performance.png")

Executing CTD-TCN LOO-CV on device: cuda


NameError: name 'labels' is not defined

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.svm import SVC

ablation_configs = [
    {"name": "Full CTD-TCN (Proposed)", "use_div": True, "use_stats": True, "tasks": ["T1", "T4", "T5"]},
    {"name": "w/o Disparity (Direct Concat)", "use_div": False, "use_stats": True, "tasks": ["T1", "T4", "T5"]},
    {"name": "w/o Handcrafted Features (Deep Only)", "use_div": True, "use_stats": False, "tasks": ["T1", "T4", "T5"]},
    {"name": "w/o Pseudo-Text Task (T1 + T4 only)", "use_div": True, "use_stats": True, "tasks": ["T1", "T4"]},
    {"name": "Handcrafted Stats Only (PCA + SVM)", "model_type": "svm"}
]

ablation_results = []

for config in ablation_configs:
    cfg_name = config["name"]
    print(f"\nEvaluating ablation setting: {cfg_name}...")

    y_true_cfg = []
    y_prob_cfg = []

    for fold in range(num_subjects):
        train_idx = [i for i in range(num_subjects) if i != fold]
        test_idx = [fold]

        lbl_tr = torch.tensor(labels[train_idx], dtype=torch.float32)
        lbl_te = labels[test_idx][0]

        if config.get("model_type") == "svm":
            s_scaler = StandardScaler().fit(data_stats[train_idx])
            s_tr = s_scaler.transform(data_stats[train_idx])
            s_te = s_scaler.transform(data_stats[test_idx])

            pca = PCA(n_components=0.90).fit(s_tr)
            svm = SVC(kernel='rbf', C=0.1, probability=True, random_state=42)
            svm.fit(pca.transform(s_tr), labels[train_idx])
            prob = svm.predict_proba(pca.transform(s_te))[0, 1]

        else:
            t1_tr, t1_te = fit_and_scale_sequences(data_t1[train_idx], data_t1[test_idx])
            t4_tr, t4_te = fit_and_scale_sequences(data_t4[train_idx], data_t4[test_idx])
            t5_tr, t5_te = fit_and_scale_sequences(data_t5[train_idx], data_t5[test_idx])

            if "T5" not in config["tasks"]:
                t5_tr = torch.zeros_like(t5_tr)
                t5_te = torch.zeros_like(t5_te)

            if config["use_stats"]:
                s_scaler = StandardScaler().fit(data_stats[train_idx])
                st_tr = torch.tensor(s_scaler.transform(data_stats[train_idx]), dtype=torch.float32)
                st_te = torch.tensor(s_scaler.transform(data_stats[test_idx]), dtype=torch.float32)
                n_stats = 30
            else:
                st_tr = torch.zeros((len(train_idx), 0), dtype=torch.float32)
                st_te = torch.zeros((1, 0), dtype=torch.float32)
                n_stats = 0

            train_ds = TensorDataset(t1_tr, t4_tr, t5_tr, st_tr, lbl_tr)
            loader = DataLoader(train_ds, batch_size=8, shuffle=True)

            model = CognitiveTaskDivergenceNet(
                in_features=5, stat_features=n_stats, hidden_dims=[32, 64, 64],
                dropout=0.25, use_divergence=config["use_div"]
            ).to(device)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
            criterion = CTDLoss(lambda_reg=0.1 if config["use_div"] else 0.0, margin=1.0)

            model.train()
            for epoch in range(45):
                for b_t1, b_t4, b_t5, b_st, b_y in loader:
                    b_t1, b_t4, b_t5, b_st, b_y = b_t1.to(device), b_t4.to(device), b_t5.to(device), b_st.to(device), b_y.to(device)
                    optimizer.zero_grad()
                    logits, delta = model(b_t1, b_t4, b_t5, b_st)
                    loss, _, _ = criterion(logits, b_y, delta)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                out_logits, _ = model(t1_te.to(device), t4_te.to(device), t5_te.to(device), st_te.to(device))
                prob = torch.sigmoid(out_logits).item()

        y_true_cfg.append(lbl_te)
        y_prob_cfg.append(prob)

    y_true_cfg = np.array(y_true_cfg, dtype=int)
    y_prob_cfg = np.array(y_prob_cfg)
    y_pred_cfg = (y_prob_cfg >= 0.5).astype(int)

    acc = accuracy_score(y_true_cfg, y_pred_cfg) * 100
    auc = roc_auc_score(y_true_cfg, y_prob_cfg)
    kappa = cohen_kappa_score(y_true_cfg, y_pred_cfg)
    mcc = matthews_corrcoef(y_true_cfg, y_pred_cfg)

    ablation_results.append({
        "Configuration": cfg_name,
        "Accuracy (%)": f"{acc:.2f}",
        "AUC-ROC": f"{auc:.4f}",
        "Cohen κ": f"{kappa:.4f}",
        "MCC": f"{mcc:.4f}"
    })

df_ablation = pd.DataFrame(ablation_results)
print("\n" + "=" * 70)
print("                    ABLATION STUDY RESULTS SUMMARY                     ")
print("=" * 70)
print(df_ablation.to_string(index=False))

df_ablation.to_csv("/content/ablation_study_results.csv", index=False)
print("\nLaTeX Code for Paper Resubmission:\n")
print(df_ablation.to_latex(index=False))